# What the anchor does, beyond the perplexities

This notebook **continues [`two_domain_walkthrough.ipynb`](two_domain_walkthrough.ipynb)** and
needs the `lfa_demo/` workspace that notebook leaves behind. Run the walkthrough first, from the
same working directory; the setup cell below stops with a message rather than a `NameError` if it
cannot find it.

The walkthrough measured perplexity: what each stage learned and what it kept. Perplexity is the
right instrument for that — it is deterministic and it reads both axes at once — but it does not
say what the models *sound* like, and it does not separate the anchor from the dose. Three
questions here:

* **Is the control fair?** The walkthrough's two unanchored controls were trained at the
  *anchored* run's number of epochs, which is past their own optimum. Section 1 re-runs each at
  its own best dose, so the part of the gap that is dose rather than anchor is measured rather
  than waved at.
* **What do the three models actually say?** Ten fixed probes, identical decoding, every
  generation printed in full, including the ones that cut against the story.
* **Does the anchor change how the new domain and the model's prior knowledge reach each other?**
  Eight more probes, in both directions, each adjacent question paired with an unrelated one so
  that transfer can be told apart from the domain having eaten the model.

Everything generated here is **qualitative**. Nothing in this package scores generated text, no
claim rests on these panels, and a different set of questions could read differently. The
walkthrough's tables are where the evidence lives.

This notebook numbers its own three sections; where it points at the walkthrough it says so. The setup cell below re-establishes what the walkthrough left on disk — the demo
paths, the recipe, the fused stage-1 checkpoint, and the five-model table section 1 reads against.
It re-scores those five models, which takes a few minutes on the card. Every cell carries the
output recorded on 2026-10-05, after the walkthrough's recording and in the same working
directory, with the paths shortened as the walkthrough's first cell describes; no number is
altered.


**Recorded outputs.** The outputs below were recorded on 2026-10-05 on one NVIDIA GeForce RTX 3090
(24 GB), with lfa 0.2.0 and torch 2.10.0+cu128, over the workspace the walkthrough's recording
left behind. Its artifact is the self-generated one at the recorded frame — 2,500 documents ×
2,048 tokens, fit at 600k samples per site, K = 32, the file `lfa init` downloads for Qwen3-0.6B —
and the supplement is on: the two controls below mix in question-and-answer pairs the model wrote
over their corpus, as the walkthrough's stages do. A run on another machine can differ, and so can
one over an artifact built there rather than downloaded; the prose around each table and panel
says what these outputs show, not which output to expect.

In [1]:
# --- continue from the walkthrough --------------------------------------------------------------
# Nothing here is new work: it re-establishes the names `two_domain_walkthrough.ipynb` left on
# disk, in the same working directory. The five-model table is re-scored because section 1 reads
# its rows against the two dose-matched controls.
from pathlib import Path

DEMO = Path("lfa_demo").resolve()
if not ((DEMO / "workspace" / "history.json").exists()
        and (DEMO / "workspace" / "artifacts" / "v1.pt").exists()):
    raise SystemExit(
        f"no LFA workspace at {DEMO}. This notebook continues "
        f"examples/two_domain_walkthrough.ipynb -- run that one first, from this working "
        f"directory, and it will leave the lfa_demo/ this cell is looking for."
    )

import dataclasses, gc, json, logging, os, sys, time, textwrap

import torch
from peft import PeftModel

import lfa
from lfa import Recipe, Workspace
from lfa.corpus import load_corpus
from lfa.evaluate import domain_perplexity, wikitext2_perplexity
from lfa.models import load_teacher, load_tokenizer

logging.basicConfig(level=logging.INFO, format="%(message)s", stream=sys.stdout, force=True)
logging.getLogger("datasets").setLevel(logging.WARNING)
logging.getLogger("lfa.evaluate").setLevel(logging.WARNING)

# The walkthrough's demo scale, unchanged -- these have to be the values that produced the runs
# on disk, not new ones.
EPOCHS = 4
N_WINDOWS = 200
DEVICE = "cuda:0"
MODEL_ID = "Qwen/Qwen3-0.6B"

DATA, WS = DEMO / "data", DEMO / "workspace"
DARWIN_TRAIN, DARWIN_HELD = DATA / "darwin" / "train", DATA / "darwin" / "heldout"
COOKERY_TRAIN, COOKERY_HELD = DATA / "cookery" / "train", DATA / "cookery" / "heldout"

artifact_path = WS / "artifacts" / "v1.pt"         # the walkthrough's copy, self-generated meta and all
artifact_v2 = WS / "artifacts" / "v2.pt"
STAGE1_FUSED = WS / "models" / "stage1_fused"      # what stage 2 adapted
exported = WS / "models" / "stage2_fused_export"   # what the walkthrough's section 9 fused

recipe = Recipe.load("qwen3-0.6b")
tokenizer = load_tokenizer(MODEL_ID)


def held_out(path):
    corpus, _ = load_corpus(path, tokenizer, max_length=recipe.sequence_length,
                            val_fraction=0.0, seed=recipe.seed,
                            keep_short_whole=recipe.keep_short_whole)
    return corpus


DOMAINS = {"A · Darwin": held_out(DARWIN_HELD), "B · cookery": held_out(COOKERY_HELD)}

MODELS = {
    "base Qwen3-0.6B":            (MODEL_ID, None),
    "after A — anchored":         (MODEL_ID, WS / "runs" / "stage1" / "final_model"),
    "after A — NO anchor":        (MODEL_ID, WS / "runs" / "stage1_unanchored" / "final_model"),
    "after A+B — anchored":       (STAGE1_FUSED, WS / "runs" / "stage2" / "final_model"),
    "after A+B — NO anchor on B": (STAGE1_FUSED, WS / "runs" / "stage2_unanchored" / "final_model"),
}


def score(base, adapter):
    model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
    if adapter is not None:
        model = PeftModel.from_pretrained(model, str(adapter))
    try:
        row = {"WikiText-2": wikitext2_perplexity(model, tokenizer, n_windows=N_WINDOWS,
                                                  device=DEVICE)}
        row.update({name: domain_perplexity(model, tokenizer, corpus, device=DEVICE)
                    for name, corpus in DOMAINS.items()})
        return row
    finally:
        del model
        gc.collect()
        torch.cuda.empty_cache()


table = {label: score(*spec) for label, spec in MODELS.items()}
base_row = table["base Qwen3-0.6B"]
print(f"lfa {lfa.__version__}; five models re-scored from {WS}")


…/lfa-anchoring/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


...

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


lfa 0.2.0; five models re-scored from /tmp/lfa-nb-run/cwd/lfa_demo/workspace


## 1. Is the control fair? Both stages, at their own dose

There is a real objection to everything the walkthrough measured, and it is worth
measuring rather than waving at.

`EPOCHS = 4` is the walkthrough's demo dose, set by its wall clock rather than read off a curve.
Both unanchored controls are trained at that same 4, because "the same run without the anchor" is
what a control is — but 4 is not *their* best dose. Without the anchor the model reaches the
domain much faster and then over-fits, and the trainer said so itself at the end of both controls
("this run trained past its own optimum"): the stage-1 control was lowest at epoch 2 (14.863) and
ended at 18.355, the stage-2 control lowest at epoch 2 (11.871) and ended at 13.190. So part of
every gap they printed is the anchor and part of it is two extra epochs of unregularized training.
The anchored curves, printed beside them below, are still falling at epoch 4 at both stages. All
four curves are printed, so the size of the effect is visible rather than asserted.

The honest version of the question is: **at each run's own best dose, what is left?** Two more
runs answer it — each control again at the epoch *its own* curve bottomed at, read off that curve
in the cell below rather than written in, with its own complete learning-rate schedule, and each
in a workspace of its own so the walkthrough's chain is untouched.

In [2]:
import dataclasses
import json as _json


def curve(run_dir):
    return [(e["epoch"], e["val_perplexity"])
            for e in _json.loads((Path(run_dir) / "training_history.json").read_text())
            if "val_perplexity" in e]


print("held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set")
for stage, runs in (("stage 1 (darwin) ", ("stage1", "stage1_unanchored")),
                    ("stage 2 (cookery)", ("stage2", "stage2_unanchored"))):
    print(f"  {stage}")
    for label, run in zip(("anchored ", "no anchor"), runs):
        print(f"     {label}  " + "  ".join(f"e{n}:{v:6.2f}" for n, v in curve(WS / "runs" / run)))

held-out perplexity by epoch, on the trainer's own 10% split of each stage's training set
  stage 1 (darwin) 
     anchored   e1: 20.39  e2: 19.57  e3: 18.78  e4: 18.65
     no anchor  e1: 16.00  e2: 14.86  e3: 16.20  e4: 18.35
  stage 2 (cookery)
     anchored   e1: 17.30  e2: 16.89  e3: 16.30  e4: 16.19
     no anchor  e1: 12.92  e2: 11.87  e3: 12.23  e4: 13.19


In [3]:
control = dataclasses.replace(recipe, lambda_qkv=0.0, lambda_mlp=0.0, mu=0.0)

# Each control's own best dose, read off ITS OWN curve above -- the rule the walkthrough's
# section 4 gives for a real run, applied to the run being asked about. Derived here rather than
# written in, so it cannot be inherited from an earlier run of this notebook.
DOSE_A = min(curve(WS / "runs" / "stage1_unanchored"), key=lambda point: point[1])[0]
DOSE_B = min(curve(WS / "runs" / "stage2_unanchored"), key=lambda point: point[1])[0]
print(f"each control's own best dose, from its own curve: stage 1 {DOSE_A} epochs, "
      f"stage 2 {DOSE_B} epochs (both anchored runs used {EPOCHS})")

t0 = time.time()

# Stage 1's control at its own dose: the base model, DOSE_A epochs, no anchor.
FAIR1 = DEMO / "workspace_dose_a"
fair1 = Workspace.init(FAIR1, MODEL_ID, artifact=str(artifact_path))
fair1.train(DARWIN_TRAIN, recipe=control, epochs=DOSE_A, device=DEVICE)

# Stage 2's control at its own dose: the SAME starting model stage 2 had (the anchored stage-1
# fused checkpoint) and the same corpus, DOSE_B epochs, no anchor. A workspace of its own, so
# the walkthrough's chain keeps pointing at its real stage 2.
FAIR2 = DEMO / "workspace_dose_b"
fair2 = Workspace.init(FAIR2, str(STAGE1_FUSED), artifact=str(artifact_v2), recipe="qwen3-0.6b")
fair2.train(COOKERY_TRAIN, recipe=control, epochs=DOSE_B, device=DEVICE)

dose_seconds = time.time() - t0
print(f"\n{dose_seconds / 60:.1f} min")
print(f"  stage-1 control, {DOSE_A} epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                          for n, v in curve(FAIR1 / "runs" / "stage1")))
print(f"  stage-2 control, {DOSE_B} epochs: " + "  ".join(f"e{n}:{v:6.2f}"
                                                          for n, v in curve(FAIR2 / "runs" / "stage1")))

each control's own best dose, from its own curve: stage 1 2 epochs, stage 2 2 epochs (both anchored runs used 4)
Artifact /tmp/lfa-nb-run/cwd/lfa_demo/workspace/artifacts/v1.pt copied to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a/artifacts/v1.pt


Default recipe for this workspace: 'qwen3-0.6b' (it names Qwen/Qwen3-0.6B)


Workspace initialised at /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_a over Qwen/Qwen3-0.6B


loader: a document that fits in one chunk is trained whole in every epoch; under keep_short_whole=False it is cut at the epoch's chunk offset like a longer one


Stage 1: Qwen/Qwen3-0.6B on /tmp/lfa-nb-run/cwd/lfa_demo/data/darwin/train (artifact v1, λ_qkv=0.0, 2 epochs)


...

  Held-out: loss=2.4791 perplexity=11.930 over 10503 tokens


Final model saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/final_model


  Training state saved (epoch=2, step=143, checkpoint=final_model)


Training history saved to /tmp/lfa-nb-run/cwd/lfa_demo/workspace_dose_b/runs/stage1/training_history.json



14.5 min
  stage-1 control, 2 epochs: e1: 15.88  e2: 15.26
  stage-2 control, 2 epochs: e1: 12.90  e2: 11.93


In [4]:
dose_a = score(MODEL_ID, FAIR1 / "runs" / "stage1" / "final_model")
dose_b = score(STAGE1_FUSED, FAIR2 / "runs" / "stage1" / "final_model")


def block(title, rows, columns):
    width = max(len(label) for label, _ in rows)
    print(f"\n{title}")
    print(f"| {'':<{width}} | " + " | ".join(f"{c:>12}" for c in columns) + " |")
    print(f"| {'-' * width} | " + " | ".join("-" * 12 for _ in columns) + " |")
    for label, row in rows:
        print(f"| {label:<{width}} | " + " | ".join(f"{row[c]:12.2f}" for c in columns) + " |")


block("stage 1 — adapting the BASE model to Darwin",
      [("base model", base_row),
       (f"anchored, {EPOCHS} epochs", table["after A — anchored"]),
       (f"no anchor, {EPOCHS} epochs", table["after A — NO anchor"]),
       (f"no anchor, {DOSE_A} epochs", dose_a)],
      ["WikiText-2", "A · Darwin"])

block("stage 2 — adapting the same after-A model to the cookbook",
      [("model it started from", table["after A — anchored"]),
       (f"anchored, {EPOCHS} epochs", table["after A+B — anchored"]),
       (f"no anchor, {EPOCHS} epochs", table["after A+B — NO anchor on B"]),
       (f"no anchor, {DOSE_B} epochs", dose_b)],
      ["WikiText-2", "A · Darwin", "B · cookery"])

Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Using the latest cached version of the dataset since wikitext couldn't be found on the Hugging Face Hub (offline mode is enabled).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).


Found the latest cached dataset configuration 'wikitext-2-raw-v1' at ~/.cache/huggingface/datasets/wikitext/wikitext-2-raw-v1/0.0.0/b08601e04326c79dfdd32d625aee71d232d685c3 (last modified on Sun May 11 17:27:57 2025).



stage 1 — adapting the BASE model to Darwin
|                     |   WikiText-2 |   A · Darwin |
| ------------------- | ------------ | ------------ |
| base model          |        17.80 |        30.12 |
| anchored, 4 epochs  |        16.06 |        19.25 |
| no anchor, 4 epochs |        26.36 |        18.84 |
| no anchor, 2 epochs |        18.93 |        15.50 |

stage 2 — adapting the same after-A model to the cookbook
|                       |   WikiText-2 |   A · Darwin |  B · cookery |
| --------------------- | ------------ | ------------ | ------------ |
| model it started from |        16.06 |        19.25 |        21.84 |
| anchored, 4 epochs    |        15.78 |        19.92 |        17.08 |
| no anchor, 4 epochs   |        24.26 |        37.47 |        14.14 |
| no anchor, 2 epochs   |        18.04 |        26.02 |        12.68 |


Read those on **all** the columns at once, which is the whole discipline of the method. The
question is never "which run has the better WikiText-2"; it is "at comparable fit on the domain
being learned, which run kept more of everything else". A control that is worse everywhere has
been beaten outright. A control that is *better* on the new domain and worse elsewhere has
bought the domain by spending what it already had — which is the trade the anchor exists to
change, not a trade it claims to abolish.

Here, at its own dose, each control fits its new domain better than the anchored run — Darwin
15.50 against 19.25, the cookbook 12.68 against 17.08 — and is worse on everything it was not
being trained on: WikiText-2 18.93 against 16.06 at stage 1, and at stage 2 WikiText-2 18.04
against 15.78 and Darwin 26.02 against 19.92. The control bought the domain by spending what it
had; the anchored run spent less and fit the domain less well.

Two things this section is honest about, whichever way the numbers fell:

* Some of the gap in the walkthrough's sections 5 and 8 is dose, not anchor. Reporting
  only the 4-epoch controls would have overstated the anchor by the difference between
  the two control rows here: the 2-epoch control is ahead of the 4-epoch one on every column
  of both blocks.
* A 2-epoch run is not simply a truncated 4-epoch run: the learning-rate schedule is laid over
  whatever number of epochs you ask for, so these are complete runs with their own warmup and
  decay. That is the right comparison, and it is also a *second* difference from the 4-epoch
  control, not a clean single-variable change.

And, still: one seed, one run per row, 193 documents.

## 2. What the three models actually say

Everything above is a perplexity. Perplexity is the right instrument for this method — it is
deterministic, it reads all three axes at once, and the walkthrough's sections 5 and 8, and section 1 above, are argued on it. What it does not tell you is what the models *sound*
like, and that is worth seeing too.

So: five probes per stage, put to three models each under identical settings — **greedy decoding,
200 new tokens, `repetition_penalty=1.15`, `no_repeat_ngram_size=6`**, the chat template with
Qwen3's thinking mode off. Two of the ten slots are deliberate repeats (`A1` asks about domain A
again after domain B has been learned; `G1` is the same general question in both panels), which
makes thirty generations from eight questions. The probe set was written down before any output
was looked at, and nothing in it was reworded, dropped or swapped afterwards. Every generation is
printed in full, including the ones that cut against the story.

**Why those two repetition controls.** Plain greedy decoding with no repetition penalty makes a
0.6B model repeat a clause until the token budget runs out, and it did: a decode of these same
ten probes recorded on 2026-09-08, with an earlier artifact and no supplement, under plain greedy
decoding, had every trained arm looping, one of them five times over. That is a property of the decoder, not of
any model, and it makes the panel unreadable; the walkthrough's section 9, which decodes without
the controls, shows it again in this recording. The penalty (the standard mild
band) and the 6-token no-repeat window are set **identically for every model**, including where
they cost one — they were chosen on whether they clear the loops, not on whose answers they
improve, and the base model pays for them too (see `G2`). Decoding stays deterministic, so there
is no seed to choose and no run to repeat until it looks good.

Three families, so it is clear what to look at:

| | |
|---|---|
| `A*`, `B*` | **adaptation** — does the model know the domain it was just trained on |
| `G*` | **retained general capability** — the axis forgetting destroys first |
| `I*` | **integration** — domain content delivered inside behaviour the base model already had: register control, brevity, doing what was asked |

**This is qualitative illustration, not evidence.** One greedy sample per model per prompt is not
a population, nothing here is scored by anything, and no claim in this notebook rests on it. The
measurements are the tables above.

One thing the second panel needs said plainly: **both** of its trained models start from the
*anchored* stage-1 checkpoint. It isolates stage-2 anchoring on its own; it is not an anchored
chain against an unanchored one.

In [5]:
# The probe set is fixed before any output is seen, and nothing below is dropped or shortened
# afterwards. `A*`/`B*` ask whether the domain was learned; `G*` ask whether general capability
# survived, which is the axis forgetting destroys first; `I*` ask for domain content delivered
# inside behaviour the base model already had -- register, brevity, following the instruction.
DARWIN_Q = "What did Darwin mean by natural selection?"
GENERAL_Q = "What is photosynthesis? Answer in two sentences."

PANEL_A = [
    ("A1", DARWIN_Q),
    ("A2", "According to The Origin of Species, what part does the struggle for existence play "
           "in the modification of species?"),
    ("G1", GENERAL_Q),
    ("G2", "List three prime numbers greater than 10."),
    ("I1", "In two sentences, explain to a ten-year-old why some animals in a species survive "
           "and others do not."),
]

PANEL_B = [
    ("A1", DARWIN_Q),                                    # the retention probe, repeated
    ("B1", "How do you make a plain suet pudding?"),
    ("B2", "What is the proper way to roast a joint of beef?"),
    ("G1", GENERAL_Q),                                   # repeated
    ("I2", "Give a short recipe for a simple soup, then explain in one sentence why boiling "
           "changes the food."),
]


# Decoding, identical for every model and every probe. Still greedy, so the section is
# deterministic and reproduces on a re-run; the two repetition controls are there because plain
# greedy decoding makes a 0.6B model repeat a clause until the budget runs out, which is a
# property of the decoder and would be easy to misread as a property of a model.
GENERATION = dict(
    max_new_tokens=200,        # unchanged -- room for a numbered recipe or a short explanation
    do_sample=False,           # greedy: no seed, no run-to-run variation, nothing to re-roll
    repetition_penalty=1.15,   # the standard mild band; 1.1 left some answers still looping
                               # (in the 2026-09-08 decode, with an earlier artifact)
    no_repeat_ngram_size=6,    # hard block on repeating a 6-token span -- short domain phrases
                               # such as "struggle for existence" still come through
)


def answer(model, question):
    """One deterministic answer, chat template, thinking off -- GENERATION for every model."""
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": question}],
                                           tokenize=False, add_generation_prompt=True,
                                           enable_thinking=False)
    ids = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(**ids, pad_token_id=tokenizer.eos_token_id, **GENERATION)
    return tokenizer.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()


def run_panel(models, probes):
    """Every model's answer to every probe. Loads one model at a time; nothing is filtered."""
    answers = {}
    for label, (base, adapter) in models.items():
        model = load_teacher(str(base), device=DEVICE, dtype=torch.bfloat16)
        if adapter is not None:
            model = PeftModel.from_pretrained(model, str(adapter))
        try:
            answers[label] = {tag: answer(model, question) for tag, question in probes}
        finally:
            del model
            gc.collect()
            torch.cuda.empty_cache()
    return answers


def show(answers, probes):
    for tag, question in probes:
        print(f"\n{'=' * 96}\n{tag}  {question}\n{'=' * 96}")
        for label, by_tag in answers.items():
            body = textwrap.fill(by_tag[tag], 92, initial_indent="    ", subsequent_indent="    ")
            print(f"\n  [{label}]\n{body}")


STAGE1_PANEL = {
    "base":       (MODEL_ID, None),
    "unanchored": (MODEL_ID, FAIR1 / "runs" / "stage1" / "final_model"),
    "anchored":   (STAGE1_FUSED, None),
}

t0 = time.time()
answers_a = run_panel(STAGE1_PANEL, PANEL_A)
print(f"stage-1 panel: 3 models x 5 probes, {time.time() - t0:.0f} s\n"
      f"  base        Qwen3-0.6B, untouched\n"
      f"  unanchored  domain A with lambda = mu = 0, at its own best dose ({DOSE_A} epochs)\n"
      f"  anchored    domain A with the recipe's anchor ({EPOCHS} epochs)")
show(answers_a, PANEL_A)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


stage-1 panel: 3 models x 5 probes, 25 s
  base        Qwen3-0.6B, untouched
  unanchored  domain A with lambda = mu = 0, at its own best dose (2 epochs)
  anchored    domain A with the recipe's anchor (4 epochs)

A1  What did Darwin mean by natural selection?

  [base]
    Charles Darwin's concept of **natural selection** is a fundamental principle in
    evolutionary biology, and it has had a profound impact on the understanding of life on
    Earth. Here’s what he meant:  ### 1. **The Origin of Species**:      Darwin proposed
    that species evolve over time through **natural selection**, where organisms with traits
    better suited to their environment are more likely to survive and reproduce. This leads
    to the gradual change in populations.  ### 2. **Survival of the Fittest**:      He
    emphasized that individuals who possess advantageous traits (e.g., physical strength,
    intelligence) have higher chances of survival and reproduction, which in turn promotes
    the spre

In [6]:
# Both stage-2 arms start from the SAME model: the anchored stage-1 checkpoint. So this panel
# isolates stage-2 anchoring alone -- it is not an anchored chain against an unanchored one.
STAGE2_PANEL = {
    "after A":    (STAGE1_FUSED, None),
    "unanchored": (STAGE1_FUSED, FAIR2 / "runs" / "stage1" / "final_model"),
    "anchored":   (exported, None),
}

t0 = time.time()
answers_b = run_panel(STAGE2_PANEL, PANEL_B)
print(f"stage-2 panel: 3 models x 5 probes, {time.time() - t0:.0f} s\n"
      f"  after A     the anchored stage-1 model, before the cookbook\n"
      f"  unanchored  domain B on top of it with lambda = mu = 0, own best dose ({DOSE_B} epochs)\n"
      f"  anchored    domain B on top of it with the anchor ({EPOCHS} epochs)")
show(answers_b, PANEL_B)

stage-2 panel: 3 models x 5 probes, 23 s
  after A     the anchored stage-1 model, before the cookbook
  unanchored  domain B on top of it with lambda = mu = 0, own best dose (2 epochs)
  anchored    domain B on top of it with the anchor (4 epochs)

A1  What did Darwin mean by natural selection?

  [after A]
    Natural selection is the process where organisms with advantageous traits are more
    likely to survive and reproduce, leading to the gradual improvement of their
    characteristics over time. This process continues until a new species develops or
    existing ones adapt to different environments.

  [unanchored]
    Natural Selection is the process that produces new species, where each has its own
    advantages. It involves selecting individuals with better traits to pass on to their
    offspring, leading to a gradual improvement in the quality of the race over time.

  [anchored]
    Natural selection is the process of producing new species, as it produces new forms. It
 

### What the probes showed, including what they did not

**The decoder first, because it changes how the rest reads.** A decode of these same ten probes
recorded on 2026-09-08, with an earlier artifact and no supplement, at the same 200 greedy tokens
but with no repetition controls, had *every* trained arm looping: that day's stage-2 anchored
model answered `B1` by repeating "Put a pound of butter in a large cask, and pour in the suet
pudding, and stir it well" five times and starting a sixth. With the settings above, none of the
thirty generations in this recording contains a repeated span of five words or longer. **The
degenerate output was the decoder.** Everything below is what the models do once it is taken
away.

**What the controls cost, stated before it is used.** `G2` is the receipt. In that 2026-09-08
decode, under plain greedy decoding, all three stage-1 models opened with "Three prime numbers
greater than 10 are". With the repetition penalty on, the base model here says "greater than
**2**" — the penalty down-weights a token the prompt already used, and the base model pays for it
like any other — and still lists 13, 17 and 19. The trained models split: the control lists 13,
17 and 29, all prime and all above 10; the anchored model says "greater than 2" as well and
answers "3, 5, and 7", none of them above 10. This recording cannot say how much of that is the
penalty and how much the training, because the new models were not decoded without it. The
settings are the price of a readable panel, and they are charged to every model equally.

**Adaptation is visible on both arms, and in the same way.** Both trained stage-1 models have
dropped the base model's register: where the base model answers `A1` and `A2` with headed,
numbered lists (`A1`'s runs out of tokens), both answer each in two sentences. The control's are
the closer to the book — variations "accumulated over time", "the improvement of each being's
characteristics", "modifying and improving each species through natural selection" — and the
anchored model's are plainer: "the gradual improvement of their characteristics over time",
"causing them to adapt and change over time". Both are right on `A1`, and neither is wrong on
`A2`. Section 1 puts the control, at its own dose, at 15.50 on Darwin and the anchored run at
19.25, against the base model's 30.12: both adapted, and the control fit the domain more closely.
The panel does not contradict that.

**On the general axis the panel does not show what the table does.** Section 1's WikiText-2
column has the control at 18.93 against 16.06 anchored, base 17.80: the control lost general
ability and the anchored run did not. These answers do not show it, and on one probe they show the
reverse. On `G2` the control is right and the anchored model wrong, as above. On `G1` neither gives
what was asked. The control gives two sentences, one of them wrong — glucose produced "through
cellular respiration", which is the process that spends it. The anchored model opens with the base
model's two sentences, nearly word for word, then adds two more the question did not ask for,
ending on photosynthesis "driving global climate change". That opening is what keeping the base
model's function looks like in text, and it brings the base model's habit of not stopping at two
sentences with it. On `I1` all three models answer in one sentence where two were asked, and both
trained arms bring in the book where the question invites it: the control's "struggle for life",
the anchored model's "struggle against natural selection".

**Stage 2: on the new domain the panel and the table agree.** Both trained arms answer the
cookbook probes in the book's period voice — "Take three pounds of suet, two eggs, and half a
pound of butter", "Take 1 pint of cream, 2 cups of sugar, and 3 eggs" — and neither answers them
well. On `B1`, of the two trained arms only the control's recipe contains suet; the anchored arm's has
none, and its last step is "then put in the pudding". On `B2`, asked how to **roast** a joint, both put it in boiling water. B ·
cookery puts the control at 12.68 against 17.08 anchored. **The anchor makes that trade; it does
not abolish it.**

On retention the panel and the table part. A · Darwin puts the control at 26.02 against 19.92
anchored: the anchored arm kept more of domain A. But asked `A1` again after the cookbook, the
control gives the better answer — "selecting individuals with better traits to pass on to their
offspring, leading to a gradual improvement in the quality of the race over time" — and the
anchored arm's never says what is selected: natural selection is "the process of producing new
species, as it produces new forms", and the rest is change over time and common ancestry. One
probe is not a population, and the table is the measurement; but this probe does not illustrate
it.

The remaining probes are ties or shared failures. `G1` in the second panel: both trained arms give
two correct sentences, the anchored arm's the more exact (chlorophyll, carbon dioxide and water,
the chloroplasts) and the control's ending "It requires water, air, and a place to grow". `I2`:
each trained arm does half of what was asked. The control gives a recipe in the period voice and an
explanation that explains nothing ("because it takes longer to cook and requires more time to
set"); the anchored arm gives no recipe at all, only the explanation, which is its starting model's
second sentence nearly word for word. The one model that does both halves is the one that has
*not* seen domain B, thinly: "mixing ingredients like chicken and vegetables".

**The count.** By the narrow criterion of answering the question asked, in the form asked, without
inventing: the anchored model is better on none of the ten and worse on three (`G2`, `B1`, and
`A1` after the cookbook). Of the other seven, three are indistinguishable (`A1` and `A2` in the
first panel, `G1` in the second) and on four both arms are weak or wrong (`G1` in the first panel,
`I1`, `B2`, `I2`). It does not adapt *better* on either domain at either stage — on both it fits
the corpus less well than the control does, and the tables said so first — and in this panel it
does not answer better either. What the tables show it keeping, these ten answers show at most in
the opening of its first `G1`.

**Where the same model appears in both panels its answers are identical** — stage 1's `anchored`
and stage 2's `after A` are the same checkpoint, and their `A1` and `G1` answers match character
for character. Decoding is deterministic, so re-running this section reproduces it.

**And the standing caveat, which cuts both ways.** Thirty single greedy generations from one 0.6B
model are an illustration, not a measurement. Nothing here is scored by anything, no claim in this
notebook rests on it, and a different set of ten questions could tell a different story. The
perplexity tables are the evidence; this is what the models sound like at this scale.


## 3. Integration, and what it should do in both directions

Everything up to here measures how much the model **keeps**. That framing makes the anchor sound
like a brake — something that stops training before it goes too far — and a brake is not very
interesting. The mechanism says something stronger than "less is lost", and this section states
what it says and then puts it to the models to see how much of it actually shows at this scale.

### What the anchor does, mechanically

One term does the work. For each sub-module of the network — an attention projection, an MLP
block — the training loss carries

> `λ · E` over `h ~ p(h)` of `‖ f_student(h) − f_teacher(h) ‖²`

Read it right to left. `f` is one sub-module, treated as a function. `h` is a *hidden state* — one
of the vectors that flows between the model's layers, which is what a sub-module actually takes as
input. `p(h)` is a distribution over those vectors, estimated from the **base** model and kept
inside the artifact the walkthrough built from the model's own text; sampling from it costs no
data and no
forward pass over anything the model once saw. So the term draws inputs the base model already
produces, pushes each through one sub-module in both the old model and the one being trained, and
charges for the difference in what comes out.

What that penalises is not weight movement as such — the much smaller `μ` term does that. It
penalises **changing what a piece of the model computes on the inputs it already sees**. A weight
change that leaves every familiar hidden state mapped where it was is free no matter how large it
is; one that re-purposes a sub-module is expensive no matter how small.

That leaves gradient descent an easy route to the new corpus and a hard one. The hard route is to
overwrite — carve out fresh machinery for the cookbook and pay for every familiar input that
machinery no longer handles the way it did. The easy route is to reach the same behaviour
*through* structure that already exists: encode "brown the meat first" using whatever the model
already has for heat, for food, for cause and effect. The anchor cannot tell the two routes apart
and does not try. It just makes the first one expensive. That is the whole of the constraint, and
the phrase for adapting under it is **guided integration**.

### Why a constraint like that predicts traffic in both directions

If new content has to be written through machinery that already exists, then the new content and
the old are not in separate compartments — they are expressed in the same representations, so each
should be reachable from the other. Two predictions follow, and they are not the same prediction:

- **Inward — prior knowledge reaching the new domain.** Ask, about the corpus's own subject,
  something the corpus does not contain. Every corpus is a slice of a subject, never the whole of
  it: *The Origin of Species* (1859) predates genetics and germ theory, and an 1853 cookbook gives
  method without chemistry. A model that has integrated the corpus should still be able to bring
  what it already knew to those questions.
- **Outward — domain semantics reaching adjacent questions.** Ask something the corpus is *not*
  about but which shares its concepts. The corpus should show up in the answer, because that is
  where the new content was written.

### Outward transfer is only a virtue if it is selective

This is the part that needs the control. Domain semantics arriving in a question that has nothing
to do with the domain is not integration — it is the domain having eaten the model, and it is the
ordinary failure mode of fine-tuning a small model on a small corpus. Section 2 did not catch
either stage-1 model doing it: on `G1` and `G2` neither brings Darwin in, and on `I1`, where the
question invites it, both do. So the probes below are the test.

So transfer on its own proves nothing, and a panel that only asked adjacent questions would be
unreadable. Each adjacent question below is therefore paired with an **unrelated** one, and the
demonstration is the *difference between the two*, not either alone. A model that draws on the
corpus for the adjacent question and leaves it out of the unrelated one has done something the
model that does it for both has not.

### The standing of this claim, stated before the outputs

The mechanism above is arithmetic: it is what the loss term computes, and that is not in question.
Whether the resulting model is *better integrated* in any measurable sense is a different question
with a worse answer. The research programme behind this package built two structural measures of
integration and neither predicted judged quality — one of them came out with the sign backwards.
So: what follows is eight questions put to six models, every generation printed in full and
uncurated. It illustrates behaviour consistent with the mechanism. It is not evidence for it,
nothing in this notebook rests on it, and a different eight questions could read differently.

| | |
|---|---|
| `P*` | **inward** — about the domain's subject, answerable only from knowledge the corpus does not hold |
| `Q*` | **outward, adjacent** — not about the corpus, but sharing its concepts; domain content here *fits* |
| `N*` | **outward control, unrelated** — shares nothing with either corpus; domain content here is a **fault** |

The eight probes were written down before any output was looked at, and none was reworded, dropped
or reordered afterwards. `Q3` is the one that reads both stages at once: it is adjacent to domain
A, and it is put to models that have since been trained on domain B. `N1` and `N2` are the same
question, so the two panels can be read against each other. Decoding is exactly section 2's —
greedy, 200 new tokens, `repetition_penalty=1.15`, `no_repeat_ngram_size=6`, the chat template with
thinking off — identical for every model and every probe, with nothing tuned per question.

In [7]:
# The transfer probes, fixed before any output was seen: nothing below was reworded, dropped or
# reordered afterwards. Same three models as section 2, the same `GENERATION` settings and the
# same helpers -- only the questions are new.
#
#   P*  inward   -- knowledge the corpus does NOT contain, asked about the corpus's own subject
#   Q*  outward  -- a question the corpus is not about, but which shares semantics with it
#   N*  control  -- a question sharing nothing with either corpus, where transfer is a FAULT
PANEL_T1 = [
    ("P1", "How does natural selection relate to genes and DNA?"),
    ("P2", "Does natural selection explain why bacteria become resistant to antibiotics?"),
    ("Q1", "Why do species introduced to a new island often multiply rapidly?"),
    ("N1", "What is the capital of Japan, and roughly how many people live there?"),
]

t0 = time.time()
answers_t1 = run_panel(STAGE1_PANEL, PANEL_T1)
print(f"stage-1 transfer panel: 3 models x 4 probes, {time.time() - t0:.0f} s\n"
      f"  P1 P2 inward   -- 1859 predates genetics and germ theory; the corpus cannot supply either\n"
      f"  Q1    outward  -- island colonisation is adjacent to the corpus, not in the question\n"
      f"  N1    control  -- unrelated; domain vocabulary appearing here is the failure mode")
show(answers_t1, PANEL_T1)

stage-1 transfer panel: 3 models x 4 probes, 22 s
  P1 P2 inward   -- 1859 predates genetics and germ theory; the corpus cannot supply either
  Q1    outward  -- island colonisation is adjacent to the corpus, not in the question
  N1    control  -- unrelated; domain vocabulary appearing here is the failure mode

P1  How does natural selection relate to genes and DNA?

  [base]
    Natural selection is a fundamental mechanism of evolution, but it operates through
    **genetic variation**, not directly through **genes** or **DNA itself**. Here's how they
    relate:  ### 1. **Genes are the basis of heredity**    - Genes (DNA sequences) carry
    instructions for building proteins, which in turn determine traits.    - These genetic
    information is passed from parents to offspring.  ### 2. **Variation occurs due to
    mutations**    - Mutations can introduce new alleles (different versions of a gene),
    leading to genetic diversity.    - This variability is crucial because it allows

In [8]:
# Stage 2, same three models as the stage-2 panel above -- all starting from the anchored
# stage-1 checkpoint. `Q3` is the one that reads both stages at once: it is adjacent to domain A,
# put to models that have since been trained on domain B.
PANEL_T2 = [
    ("P3", "Why does browning meat before stewing improve the flavour?"),
    ("Q2", "How did households keep food from spoiling before refrigeration?"),
    ("N2", "What is the capital of Japan, and roughly how many people live there?"),
    ("Q3", "In a place with little food, what happens to a population of animals over many "
           "generations?"),
]

t0 = time.time()
answers_t2 = run_panel(STAGE2_PANEL, PANEL_T2)
print(f"stage-2 transfer panel: 3 models x 4 probes, {time.time() - t0:.0f} s\n"
      f"  P3 inward   -- an 1853 cookbook gives method, not the chemistry of why it works\n"
      f"  Q2 outward  -- adjacent to domain B\n"
      f"  N2 control  -- the same unrelated question as N1, so the two panels can be compared\n"
      f"  Q3 outward  -- adjacent to domain A, asked after domain B was learned")
show(answers_t2, PANEL_T2)

stage-2 transfer panel: 3 models x 4 probes, 14 s
  P3 inward   -- an 1853 cookbook gives method, not the chemistry of why it works
  Q2 outward  -- adjacent to domain B
  N2 control  -- the same unrelated question as N1, so the two panels can be compared
  Q3 outward  -- adjacent to domain A, asked after domain B was learned

P3  Why does browning meat before stewing improve the flavour?

  [after A]
    The brownish colouration of the meat, which is a result of oxidation and heat, enhances
    its flavor by making it more attractive to taste buds. This process also helps in
    preserving the meat better during storage.

  [unanchored]
    Because it allows the flavor to penetrate and enhance the taste.

  [anchored]
    Because it gives a better flavor to the meat, and prevents the loss of its natural
    taste.

Q2  How did households keep food from spoiling before refrigeration?

  [after A]
    Households kept food in sealed containers, stored away for a long time, and used the
 

### What the panel showed, and what it did not

**Outward transfer separates the two arms once, at stage 2, and in the anchored arm's favour.**
The design above rests on pairs: an adjacent question, where domain content fits, and an unrelated
one, where it is a fault.

`N1` and `N2`, the unrelated control, show no fault of that kind anywhere. Every trained model
names Tokyo and gives 10 or 14 million people, in one or two sentences, and no Darwin and no
cookery arrive in any of them. The stage-2 anchored arm does slip on a fact: "Approximately 10
million people reside in Japan", the country where it meant the city. The **base** model is the
one that gets `N1` wrong outright, with "Osaka City", so the control question cannot be read as
the trained models degrading — at this size the base model is not a gold standard either.

`Q1`, the adjacent question `N1` was paired with, draws on domain A on both arms, and neither arm
uses it well. The control brings natural selection in backwards — introduced species spread "due
to the lack of natural selection for their own improvement" — and credits it to "the passage",
where none was given. The anchored model uses the book's "conditions of life" and contradicts
itself: the island's conditions "are not as favorable for their survival", and the species
multiply anyway. The base model lists six generic factors, from "Adaptation and Survival" to
"Environmental Factors"; its "limited competition" is the nearest any of the three comes to an
answer. As a pair, `N1` and `Q1` show both arms selective — domain vocabulary in the adjacent
question, none in the unrelated one — and neither using it well.

`Q2`, adjacent to domain B, shows transfer on both arms. Asked how food was kept before
refrigeration, the model that has **not** seen the cookbook answers vaguely about sealed
containers and "the same method as now". Both trained arms say food was kept "in a dry place". The
anchored arm stops there, in one thin sentence. The control goes on in the cookbook's imperative
voice into something else — *"If the weather is warm, you should have an iron kettle with a tight
cover for boiling water"* — and does not come back to preserving.

`Q3` reads both stages at once — a domain-A-adjacent question put to models that have since been
trained on the cookbook — and it is the one probe in this section that separates the arms. The
model before the cookbook answers from a passage it was never given ("The passage suggests") that
better-adapted species might persist. The control has lost the question to the cookbook's
household advice: *"A small family will soon become large; and if they are kept in the same house
for years, it is very dangerous. If you have an old horse that has been used by several families,
it should be well mended before it is given away."* That is a domain carrying a question off,
arriving on an adjacent question rather than the unrelated one. The anchored arm answers in one
sentence from domain A — *"The population is kept in the same condition by the natural selection
process"* — vague, but about the question, and the string "natural selection" occurs in no other
answer to it.

**Inward transfer shows weakly at stage 1, on both arms, and stage 2 cannot test it at this
scale.** On `P1` and `P2` the base model knows genetics and antibiotic resistance, so there is
something to transfer. On `P1` the control keeps more of it: natural selection "acts on the
genetic makeup of organisms", and "variations in gene expression" are passed on — muddled, since
expression is not what is inherited — with a horse's coat colour as its example. The anchored
model answers in the terms of heredity alone ("the variation of traits, which are inherited by
offspring") and mentions neither genes nor DNA. On `P2` both say yes and keep a trace — the
control's "genetic variation and inheritance", the anchored model's "accumulated mutations over
time" — and neither gives the account the base model does, of the mutated bacteria surviving and
reproducing. Neither arm says as much as the base model on either probe.

Both arms also answer from a text that was never given: the control's `P2` and `Q1` cite "the
passage", the anchored model's `P2` says "according to the text", and the same anchored checkpoint,
as stage 2's "after A", opens `Q3` with "The passage suggests". The question-and-answer supplement
both arms trained with is written over passages of the corpus, which is a likely source; nothing
here tests that.

`P3` fails to test anything, and that is worth saying plainly rather than reading a result into
it. Asked why browning meat improves flavour, **none** of the three models knows the chemistry —
not even "after A", which has never seen a cookbook and is the closest thing here to an untouched
control. It credits "oxidation and heat" and "making it more attractive to taste buds"; the control
says browning "allows the flavor to penetrate and enhance the taste", and the anchored arm that it
"prevents the loss of its natural taste". Prior knowledge cannot transfer inward if the model never
had it, and a 0.6B model does not have the Maillard reaction. The probe measures the model's size,
not the method.

**The summary, direction by direction.** Outward transfer and its selectivity: **one difference
between the arms, on `Q3`** — no model brings a domain into the unrelated question, both bring
domain A to `Q1` and the cookbook to `Q2`, and on `Q3` the control is carried off by the cookbook
while the anchored arm answers from Darwin. Inward transfer: **weakly visible at stage 1, on both
arms**, the control keeping more of the base model's genetics on `P1`; **not testable at stage 2**,
because the prior knowledge `P3` asks for is not in a model this small. And the standing caveat is
unchanged by any of it — twenty-four greedy generations are an illustration of a mechanism, not a
measurement of one, and the tables above are still where the evidence in this notebook lives.


## Where that leaves the pair of notebooks

Section 1 split the walkthrough's gaps into dose and anchor by giving each control its own best
number of epochs. At that dose each control fits its new domain more closely than the anchored run
and keeps less of what it was not trained on: WikiText-2 at both stages, and domain A across
stage 2. Sections 2 and 3 put the models to fixed questions, and at this scale the answers mostly do
not show that difference. The two arms answer most probes alike or equally weakly; the control's
answers are the better ones on `G2`, on `B1` and on the Darwin probe after the cookbook, and in
section 3 it keeps more of the base model's genetics on `P1`. The anchored model's one clear
advantage is `Q3`, a Darwin-adjacent question after the cookbook, which it answers from Darwin
while the control's answer turns into household advice. In this recording the anchor shows in the
perplexities, and in the generations on that one probe.

And the caveat that has stood over every panel here: one seed, one run per row, single greedy
generations from one 0.6B model. These are illustrations of a mechanism, not measurements of one.
The walkthrough's tables are the evidence; this is what the models sound like at this scale.

Back to [`two_domain_walkthrough.ipynb`](two_domain_walkthrough.ipynb) for the flow itself, or to
`docs/concepts.md` for what the anchor does and how a run is read.
